# Pipeline metagenómico (HPC)

**Este notebook está diseñado para correr en un clúster de cómputo de alto rendimiento, no en un
portátil.** A diferencia de versiones anteriores, todo el flujo -- detección dual de ARG, su
posición y su vinculación a especie/posición genómica -- corre automáticamente por muestra, sin
pasos manuales diferidos. El costo dominante es una descarga única y compartida (no por muestra) de
la base `nt` completa de NCBI para BLAST local (cientos de GB) -- ver la sección "Base BLAST local"
en la sección 2. Para una versión liviana, de laptop, sin esa base, usa `03_pipeline_inicial.ipynb`;
para las mediciones de recursos que fundamentan este diseño, `02_estimacion_recursos.ipynb`.

Toma uno o más `run_accession` de ENA (proyecto [PRJEB11755](https://www.ebi.ac.uk/ena/browser/view/PRJEB11755)),
descarga las lecturas crudas, las procesa hasta obtener contigs ensamblados, y responde, **automáticamente
por cada muestra** (sección 4):

1. **¿Qué genes de resistencia antimicrobiana (ARG) hay?** — detección dual, en paralelo: [CARD](https://card.mcmaster.ca/)
   + RGI, y [NCBI AMRFinderPlus](https://github.com/ncbi/amr), cruzadas por gen y por familia/clase de droga.
2. **¿Dónde, dentro de su contig?** — coordenadas, cadena y posición relativa (0-1) de cada ARG, para las dos
   herramientas.
3. **¿A qué especie pertenece, y en qué posición de su genoma de referencia?** — BLAST **local** (`blastn`
   contra una copia local de `nt`, restringida a taxonomía Bacteria) sobre los contigs con ARG: especie del
   mejor hit, y coordenadas del ARG proyectadas sobre esa secuencia de referencia (accession de NCBI).
4. **¿Qué se sabe de esa cepa/especie?** — vía [BacDive](https://bacdive.dsmz.de/), en `05_modelo_ml.ipynb`
   (se consulta solo sobre las especies de los genes que el modelo termine señalando, no por cada muestra aquí).


## 1. Requisitos de cómputo, tiempo y almacenamiento

Las mediciones detalladas por paso -- RAM, disco, tiempo de cómputo local y volumen de descarga --, tanto para una sola muestra como para el lote completo definido en `RUNS` (sección 2 de este notebook), se movieron a **`02_estimacion_recursos.ipynb`**. Se separaron de aquí para que las cifras salgan de una corrida real, instrumentada paso a paso sobre una muestra nunca antes descargada (no de una sola corrida de referencia hecha a mano), combinada con una consulta en vivo a la API de ENA para capturar cuánto varía el tamaño real entre muestras -- y para no mezclar la fase de planeación de recursos con el pipeline en sí.

Orden de magnitud esperado (ese notebook trae el detalle, la metodología y el cálculo real; correlo para tener cifras propias): una muestra individual con `SUBSAMPLE=1_000_000` ronda ~7.5GB de disco transitorio y ~30 min de cómputo local, más la descarga (comprimida, variable según la muestra); el lote de 50 muestras configurado abajo descarga en total dos órdenes de magnitud más que eso y no requiere más disco pico que el de una sola muestra en ningún momento, porque `procesar_muestra()` limpia cada muestra antes de empezar la siguiente (sección 3.5).

## 2. Configuración, parámetros y utilidades

Comenzamos importando las librerías necesarias, y definir los parámetros de ejecución del pipeline. En la variable ´RUNS´ definimos cuáles muestras de la base de datos de ENA serán descargadas y procesadas por el pipeline. Debido a que este pipeline está diseñado para computadores portátiles comerciales, utilizamos una estrategia de submuestreo para crear muestras aleatorias de un tamaño fijo, debido a que procesar una muestra entera puede ser significativamente intenso para computadores personales. Para el ejemplo de prueba, creamos submuestras de 1 millones de pares de bases.  

In [ ]:
import gzip
import hashlib
import re
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from Bio import SeqIO
import os

from seleccion_muestras import seleccionar_runs

# ---- Muestras a procesar ----
# Se consultan en vivo a la API de ENA, no se hardcodean IDs: ver seleccion_muestras.py
# para el criterio (reparto proporcional por grupo del proyecto, favoreciendo las
# corridas mas livianas dentro de cada grupo).
RUNS = seleccionar_runs(n_muestras=50)
print(f"{len(RUNS)} muestras seleccionadas de PRJEB11755")

# ---- Parámetros del pipeline ----
SUBSAMPLE = 1_000_000   # nº de pares de lecturas a submuestrear (ensayo rápido; sube o quita el paso para la muestra completa)
THREADS   = 12            # hilos de CPU para fastp / megahit / rgi / amrfinder / blastn
SEED      = 100          # semilla fija -> submuestreo reproducible
CARD_URL  = "https://card.mcmaster.ca/latest/data"

# Credenciales de BacDive (registro gratuito en https://bacdive.dsmz.de/api/bacdive/registration/register/).
# Se leen de variables de entorno para no dejar contraseñas en el notebook.
# NOTA: BacDive ya no se consulta desde este notebook -- se movera a 05_modelo_ml.ipynb,
# para correrla solo sobre los genes que el modelo termine senalando, no por cada especie
# de cada muestra procesada aqui.

BACDIVE_USER     = os.environ.get("BACDIVE_EMAIL", "")
BACDIVE_PASSWORD = os.environ.get("BACDIVE_PASSWORD", "")

Esta función es estética. Ejecutamos cada una de las consultas del pipeline con esta función, para que el Notebook muestre el progreso en vivo de la consulta, asegurando al usuario que la consulta está siendo procesada, o mostrar errores inmediatamente si se presentan.

In [2]:
def sh(cmd, live=False):
    """Ejecuta un comando de shell y aborta si falla.

    Con live=True muestra la salida linea a linea con el tiempo transcurrido (util para
    pasos largos como el ensamblado); si no, la muestra completa al terminar.
    """
    print(f"$ {cmd}" + ("\n" if live else ""))
    if not live:
        r = subprocess.run(cmd, shell=True, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if r.stdout:
            print(r.stdout)
        if r.returncode != 0:
            raise RuntimeError(f"Fallo (codigo {r.returncode}): {cmd}")
        return r.stdout

    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        el = time.time() - t0
        print(f"[{int(el // 60):>2}m{int(el % 60):02d}s] {line}", end="", flush=True)
    proc.wait()
    el = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"Fallo (codigo {proc.returncode}) tras {int(el)}s: {cmd}")
    print(f"\nCompletado en {int(el // 60)}m{int(el % 60):02d}s")

### Base de datos CARD (una sola vez, compartida por todas las muestras)

Estrategia en cascada: si ya está cargada la usa; si no, reutiliza un `card.json` local; y si tampoco, lo descarga.

In [3]:
def ensure_card_db():
    if Path("localDB/card.json").exists():
        print("CARD ya esta cargada en localDB/")
        return

    if Path("card.json").exists():
        print("Cargando card.json en la raíz")
    elif Path("data/card.json").exists():
        print("Reusar card.json ya existente.")
        sh("cp data/card.json card.json")
    else:
        print("No hay CARD en ningun sitio; descargando...")
        sh(f"wget -q {CARD_URL} -O card_data.tar.bz2")
        sh("tar -xjf card_data.tar.bz2 ./card.json")

    sh("rgi load --card_json card.json --local")
    assert Path("localDB/card.json").exists(), "CARD no quedó cargada; revisa los pasos de arriba."
    print("Base CARD lista en:", Path("localDB").resolve())

ensure_card_db()

CARD ya esta cargada en localDB/


### Base de datos NCBI AMRFinderPlus (una sola vez, compartida)

Deteccion de ARG en paralelo a CARD/RGI (seccion 3.4). A diferencia de `localDB/card.json`, esta
base pesa demasiado para versionarla en git (~250MB), asi que `localDB/amrfinderplus/` esta en
`.gitignore` y cada quien la descarga localmente.

In [ ]:
AMRFINDERPLUS_DB_DIR = Path("localDB/amrfinderplus")


def ensure_amrfinderplus_db():
    if (AMRFINDERPLUS_DB_DIR / "latest").exists():
        print("AMRFinderPlus ya tiene base local en", AMRFINDERPLUS_DB_DIR)
        return
    AMRFINDERPLUS_DB_DIR.mkdir(parents=True, exist_ok=True)
    sh(f"amrfinder_update -d {AMRFINDERPLUS_DB_DIR}")
    assert (AMRFINDERPLUS_DB_DIR / "latest").exists(), "AMRFinderPlus DB no quedó lista; revisa el paso de arriba."
    print("Base AMRFinderPlus lista en:", AMRFINDERPLUS_DB_DIR.resolve())


ensure_amrfinderplus_db()
AMRFINDER_DB = AMRFINDERPLUS_DB_DIR / "latest"

### Descarga robusta con verificación MD5

Reanuda si la descarga se corta y valida el checksum contra el que reporta ENA; una descarga corrupta causa errores
fantasma más adelante, así que se aborta si no coincide.

In [4]:
def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}TB"


def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(url, out, want_md5, chunk=1 << 18, max_retries=1000):
    """Descarga robusta: progreso, reanudacion tras cortes y verificacion MD5."""
    out = Path(out)
    if out.exists() and md5sum(out) == want_md5:
        print(f"Ya existe y el MD5 coincide: {out.name}")
        return
    attempt = 0
    last_print = 0.0
    while True:
        existing = out.stat().st_size if out.exists() else 0
        headers = {"Range": f"bytes={existing}-"} if existing else {}
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as resp:
                if existing and getattr(resp, "status", 200) == 206:
                    mode = "ab"
                    total = existing + int(resp.headers.get("Content-Length", 0))
                else:
                    existing, mode = 0, "wb"
                    total = int(resp.headers.get("Content-Length", 0))
                done, t0 = existing, time.time()
                with open(out, mode) as f:
                    while True:
                        block = resp.read(chunk)
                        if not block:
                            break
                        f.write(block)
                        done += len(block)
                        now = time.time()
                        if now - last_print >= 0.2 or (total and done >= total):
                            last_print = now
                            el = now - t0
                            spd = (done - existing) / el if el > 0 else 0
                            pct = (done / total * 100) if total else 0
                            eta = (total - done) / spd if spd > 0 else 0
                            print(f"\r  {out.name}: {pct:5.1f}%  {_human(done)}/{_human(total)}  "
                                  f"{_human(spd)}/s  ETA {int(eta // 60)}m{int(eta % 60):02d}s   ",
                                  end="", flush=True)
            if total and done < total:
                raise IOError(f"conexion cerrada antes de tiempo: {_human(done)}/{_human(total)}")
            print()
            break
        except urllib.error.HTTPError as e:
            if e.code == 416:
                print(f"\n  Rango invalido; reinicio {out.name} desde cero.")
                out.unlink(missing_ok=True)
                continue
            raise
        except Exception as e:
            attempt += 1
            if attempt > max_retries:
                raise
            print(f"\n  Interrumpido ({e}); reintentando en 5s [{attempt}]...", flush=True)
            time.sleep(5)
    got = md5sum(out)
    if got != want_md5:
        raise RuntimeError(f"MD5 no coincide para {out.name}: {got} != {want_md5}")
    print(f"  Descarga verificada: {out.name}")

### Base BLAST local (`nt` completa de NCBI, restringida a Bacteria) -- una sola vez, compartida

**La descarga más pesada del pipeline: cientos de GB.** Es justo la razón de ser de este notebook
frente a la versión de laptop -- se paga una vez, por adelantado, para que cada muestra después
consulte contra una copia local en vez de hacer una llamada remota por contig a los servidores de
NCBI (lento, con límite de tasa, y proporcional al número de muestras). `update_blastdb.pl` descarga
la base ya preformada para `blastn` (no hace falta `makeblastdb`), más `taxdb` (pequeña, un par de
cientos de MB) que es la que permite filtrar por taxonomía con `-taxids`.

Se restringe cada búsqueda a Bacteria (`-taxids 2`, sección 3.7): los ARG que este pipeline detecta
son de origen bacteriano casi siempre, y restringir el espacio de búsqueda dentro de `nt` acelera
bastante cada `blastn` sin perder cobertura real para este caso.

In [ ]:
BLASTDB_DIR = Path("localDB/blastdb")
NT_DB = BLASTDB_DIR / "nt"
TAXID_BACTERIA = "2"  # taxid de NCBI para el dominio Bacteria


def ensure_blast_nt_db():
    BLASTDB_DIR.mkdir(parents=True, exist_ok=True)

    if (BLASTDB_DIR / "nt.nal").exists() or (BLASTDB_DIR / "nt.nin").exists():
        print("nt ya esta descargada en", BLASTDB_DIR)
    else:
        print("Descargando la base nt completa de NCBI (cientos de GB) -- puede tardar horas "
              "incluso en una red rapida de HPC. No se interrumpe: update_blastdb.pl reanuda solo.")
        sh(f"cd {BLASTDB_DIR} && update_blastdb.pl --decompress --source ncbi nt", live=True)

    if (BLASTDB_DIR / "taxdb.btd").exists():
        print("taxdb ya esta descargada en", BLASTDB_DIR)
    else:
        print("Descargando taxdb (necesaria para filtrar por taxonomia con -taxids)...")
        sh(f"cd {BLASTDB_DIR} && update_blastdb.pl --decompress taxdb")

    os.environ["BLASTDB"] = str(BLASTDB_DIR.resolve())
    print("Base BLAST local lista en:", BLASTDB_DIR.resolve())


ensure_blast_nt_db()


def organismo_de_descripcion(desc):
    """Extrae 'Genero especie' del mejor hit, de forma conservadora."""
    desc = desc.replace("[", "").replace("]", "")
    desc = re.sub(r"^(PREDICTED:|UNVERIFIED:|MAG:|TPA:|TPA_asm:)\s*", "", desc).strip()
    palabras = desc.split()
    if not palabras:
        return None
    if palabras[0].lower() in ("uncultured", "unidentified", "bacterium", "synthetic"):
        return palabras[0].lower()
    if len(palabras) >= 2 and palabras[1][0].islower():
        return f"{palabras[0]} {palabras[1]}"
    return palabras[0]


_BLAST_COLS = ["qseqid", "sacc", "slen", "qstart", "qend", "sstart", "send", "pident", "length", "stitle"]


def blast_local_taxonomia_y_posicion(contigs_recs, out_dir, tag):
    """blastn LOCAL contra la copia local de nt (restringida a Bacteria): especie del mejor hit,
    mas las coordenadas del alineamiento (en el contig y en la secuencia de referencia de NCBI)
    para poder proyectar despues la posicion de cada ARG sobre esa referencia.

    Una sola llamada a blastn para TODOS los contigs a la vez (multi-FASTA), no una por contig --
    a diferencia del BLAST remoto (limitado por la tasa de NCBI), localmente blastn carga la base
    una vez y recorre todas las queries, mucho mas eficiente que repetir la carga por contig."""
    vacio = {"taxon": None, "identidad_%": None, "descripcion": "sin hits", "hit_accession": None,
             "hit_len_ref": None, "query_start": None, "query_end": None,
             "sbjct_start": None, "sbjct_end": None}
    if not contigs_recs:
        return pd.DataFrame(columns=["contig", *vacio.keys()])

    query_fa = out_dir / f"_blast_query_{tag}.fasta"
    out_tsv = out_dir / f"_blast_out_{tag}.tsv"
    with open(query_fa, "w") as f:
        for r in contigs_recs:
            f.write(f">{r.id}\n{r.seq}\n")

    sh(f"blastn -query {query_fa} -db {NT_DB} -task megablast -taxids {TAXID_BACTERIA} "
       f"-max_target_seqs 1 -max_hsps 1 -outfmt \"6 {' '.join(_BLAST_COLS)}\" "
       f"-num_threads {THREADS} -out {out_tsv}")

    if out_tsv.exists() and out_tsv.stat().st_size > 0:
        hits = (pd.read_csv(out_tsv, sep="\t", names=_BLAST_COLS)
                 .groupby("qseqid", as_index=False).first().set_index("qseqid"))
    else:
        hits = pd.DataFrame(columns=_BLAST_COLS).set_index("qseqid")
    query_fa.unlink(missing_ok=True)
    out_tsv.unlink(missing_ok=True)

    filas = []
    for r in contigs_recs:
        if r.id not in hits.index:
            filas.append({"contig": r.id, **vacio})
            continue
        h = hits.loc[r.id]
        filas.append({"contig": r.id, "taxon": organismo_de_descripcion(str(h["stitle"])),
                      "identidad_%": round(float(h["pident"]), 1), "descripcion": str(h["stitle"])[:70],
                      "hit_accession": h["sacc"], "hit_len_ref": int(h["slen"]),
                      "query_start": int(h["qstart"]), "query_end": int(h["qend"]),
                      "sbjct_start": int(h["sstart"]), "sbjct_end": int(h["send"])})
    return pd.DataFrame(filas)


def proyectar_posicion_en_referencia(row):
    """Aproxima donde cae el ARG (Start/Stop del contig) dentro de la secuencia de referencia de
    NCBI (hit_accession), interpolando linealmente sobre el tramo que BLAST alineo (query_start-
    query_end -> sbjct_start-sbjct_end). Es una aproximacion -- ignora indels dentro del
    alineamiento -- y solo es valida si el ARG cae dentro de ese tramo alineado; si no, devuelve
    None en vez de extrapolar."""
    qs, qe = row.get("query_start"), row.get("query_end")
    ss, se = row.get("sbjct_start"), row.get("sbjct_end")
    inicio, fin = row.get("Start"), row.get("Stop")
    if pd.isna(qs) or pd.isna(ss) or pd.isna(inicio) or pd.isna(fin):
        return pd.Series({"ref_pos_inicio": None, "ref_pos_fin": None})
    if not (qs <= inicio <= qe and qs <= fin <= qe):
        return pd.Series({"ref_pos_inicio": None, "ref_pos_fin": None})  # ARG fuera del tramo alineado

    def proyectar(pos):
        frac = (pos - qs) / (qe - qs) if qe != qs else 0
        return round(ss + frac * (se - ss))

    return pd.Series({"ref_pos_inicio": proyectar(inicio), "ref_pos_fin": proyectar(fin)})

### BacDive (rasgos por especie/cepa)

Se consulta **una vez por especie distinta** detectada (no por gen), y solo si hay credenciales configuradas arriba.
Si no hay credenciales, el pipeline sigue funcionando: simplemente no agrega la columna de rasgos.

In [6]:
BACDIVE_FIELDS = {
    "bacdive_id": ("General", "BacDive-ID"),
    "ncbi_tax_id": ("General", "NCBI tax id", "NCBI tax id"),
    "gram_stain": ("Morphology", "cell morphology", "gram stain"),
    "oxygen_tolerance": ("Physiology and metabolism", "oxygen tolerance", "oxygen tolerance"),
}


def _get_nested(d, path):
    for key in path:
        if isinstance(d, list):
            d = d[0] if d else {}
        if not isinstance(d, dict):
            return None
        d = d.get(key)
    return d


def bacdive_traits(taxon):
    """Consulta BacDive por 'Genero especie' y devuelve un resumen minimo de rasgos de la primera cepa encontrada."""
    if not (BACDIVE_USER and BACDIVE_PASSWORD):
        return {"taxon": taxon, "bacdive_id": None, "ncbi_tax_id": None,
                "gram_stain": None, "oxygen_tolerance": None,
                "bacdive_nota": "sin credenciales BACDIVE_EMAIL/BACDIVE_PASSWORD"}
    if not taxon or len(taxon.split()) < 2:
        return {"taxon": taxon, "bacdive_id": None, "ncbi_tax_id": None,
                "gram_stain": None, "oxygen_tolerance": None,
                "bacdive_nota": "taxon sin resolucion de especie, no se consulta"}

    import bacdive
    try:
        client = bacdive.BacdiveClient(BACDIVE_USER, BACDIVE_PASSWORD)
        genero, especie = taxon.split()[:2]
        n = client.search(taxonomy=f"{genero} {especie}")
        if not n:
            return {"taxon": taxon, "bacdive_id": None, "ncbi_tax_id": None,
                    "gram_stain": None, "oxygen_tolerance": None, "bacdive_nota": "sin resultados en BacDive"}
        entry = next(client.retrieve())
        fila = {"taxon": taxon, "bacdive_nota": None}
        for campo, path in BACDIVE_FIELDS.items():
            fila[campo] = _get_nested(entry, path)
        return fila
    except Exception as e:
        return {"taxon": taxon, "bacdive_id": None, "ncbi_tax_id": None,
                "gram_stain": None, "oxygen_tolerance": None, "bacdive_nota": f"error: {e}"}

## 3. Pipeline por muestra

Cada paso es idempotente (si el archivo de salida ya existe, se omite), así que puedes volver a correr el notebook
sin repetir trabajo costoso. Como la sección 3.6 borra `raw/<run>/` y todo `work/<run>/` al terminar
cada muestra, la marca de "ya esta lista" para saltarse una muestra por completo pasa a ser `results/<run>/resistoma_<run>.csv`
en vez de esos archivos borrados.

### 3.1 Resolver URLs y checksums en ENA, y descargar

In [7]:
def resolver_y_descargar(run, raw_dir):
    ena_url = (
        "https://www.ebi.ac.uk/ena/portal/api/filereport"
        f"?accession={run}&result=read_run"
        "&fields=fastq_ftp,fastq_md5&format=tsv"
    )
    ena = pd.read_csv(ena_url, sep="\t")
    ftp = ena.loc[0, "fastq_ftp"].split(";")
    md5 = ena.loc[0, "fastq_md5"].split(";")
    url1, url2 = "https://" + ftp[0], "https://" + ftp[1]
    r1 = raw_dir / f"{run}_1.fastq.gz"
    r2 = raw_dir / f"{run}_2.fastq.gz"

    download(url1, r1, md5[0])
    download(url2, r2, md5[1])
    return r1, r2

### 3.2 Control de calidad (fastp) y submuestreo reproducible (seqtk)

In [8]:
def qc_y_submuestreo(run, r1, r2, work_dir, out_dir):
    clean1, clean2 = work_dir / "clean_1.fastq.gz", work_dir / "clean_2.fastq.gz"
    if clean1.exists() and clean2.exists():
        print("QC ya realizado, se omite")
    else:
        sh(f"fastp -i {r1} -I {r2} -o {clean1} -O {clean2} "
           f"-w {THREADS} -h {out_dir}/{run}_fastp.html -j {out_dir}/{run}_fastp.json")

    sub1, sub2 = work_dir / "sub_1.fastq", work_dir / "sub_2.fastq"
    if sub1.exists() and sub2.exists():
        print("Submuestreo ya realizado, se omite")
    else:
        sh(f"seqtk sample -s{SEED} {clean1} {SUBSAMPLE} > {sub1}")
        sh(f"seqtk sample -s{SEED} {clean2} {SUBSAMPLE} > {sub2}")
    return sub1, sub2

### 3.3 Ensamblado (MEGAHIT) y predicción de genes (Prodigal)

In [ ]:
def ensamblar_y_predecir_genes(sub1, sub2, work_dir):
    contigs = work_dir / "megahit_out" / "final.contigs.fa"
    if contigs.exists():
        print("Ensamblado ya existe, se omite")
    else:
        shutil.rmtree(work_dir / "megahit_out", ignore_errors=True)  # MEGAHIT exige que NO exista
        sh(f"megahit -1 {sub1} -2 {sub2} -t {THREADS} -o {work_dir}/megahit_out", live=True)

    genes_faa, genes_fna = work_dir / "genes.faa", work_dir / "genes.fna"
    # GFF (formato prodigal): lo necesita AMRFinderPlus (seccion 3.4) para reportar posicion.
    genes_gff = work_dir / "genes.gff"
    if not genes_faa.exists():
        sh(f"prodigal -i {contigs} -a {genes_faa} -d {genes_fna} -p meta -q -o {genes_gff} -f gff")
    n_genes = int(subprocess.run(f"grep -c '>' {genes_faa}", shell=True,
                                 text=True, capture_output=True).stdout or 0)
    print(f"Genes predichos: {n_genes}")
    return contigs, genes_faa, genes_gff

### 3.4 Detección de genes de resistencia: CARD/RGI y NCBI AMRFinderPlus, en paralelo

Las dos corren sobre las mismas proteínas predichas por `prodigal` -- ninguna reemplaza a la otra,
se cruzan por gen y por familia/clase de droga para ver dónde coinciden y dónde cada una encuentra
algo que la otra no (heurístico, ver `cruzar_card_amrfinder`). AMRFinderPlus corre en modo combinado
(proteína + nucleótido + GFF, no solo proteína): es la única forma de que reporte posición
(`Contig id`/`Start`/`Stop`/`Strand`), que la sección 3.5 necesita igual que ya usa las de RGI.

In [ ]:
def detectar_arg(run, genes_faa, out_dir):
    if not (Path("localDB/card.json").exists() or Path("card.json").exists()):
        raise RuntimeError(
            "No encuentro la base CARD local. Corre antes ensure_card_db()."
        )
    rgi_out = out_dir / f"rgi_{run}"
    if not Path(f"{rgi_out}.txt").exists():
        sh(f"rgi main -i {genes_faa} -o {rgi_out} -t protein -a DIAMOND --local --clean")

    rgi = pd.read_csv(f"{rgi_out}.txt", sep="\t")
    print(f"Genes de resistencia detectados (CARD/RGI): {len(rgi)}")
    print("Por tipo de acierto (Cut_Off):")
    print(rgi["Cut_Off"].value_counts())
    return rgi


def detectar_arg_amrfinder(run, contigs, genes_faa, genes_gff, out_dir):
    amr_out = out_dir / f"amrfinder_{run}.tsv"
    if not amr_out.exists():
        sh(f"amrfinder -p {genes_faa} -n {contigs} -g {genes_gff} -a prodigal "
           f"-d {AMRFINDER_DB} -o {amr_out} --threads {THREADS}")

    amrfinder = pd.read_csv(amr_out, sep="\t")
    print(f"Genes de resistencia detectados (AMRFinderPlus): {len(amrfinder)}")
    return amrfinder


def normalizar_gen(nombre):
    """Simbolo de gen/familia a una forma comparable: mayusculas, sin simbolos ni espacios."""
    return re.sub(r"[^A-Z0-9]", "", str(nombre).upper())


def mismo_gen(a, b, min_len=4):
    """Considera que dos simbolos nombran lo mismo si, normalizados, uno contiene al otro --
    CARD y AMRFinderPlus no siempre comparten prefijo de familia (p. ej. 'TEM-181' vs 'blaTEM-181').
    Con cadenas muy cortas (< min_len) exige igualdad exacta para no cruzar cualquier cosa."""
    a, b = normalizar_gen(a), normalizar_gen(b)
    if not a or not b:
        return False
    if len(a) < min_len or len(b) < min_len:
        return a == b
    return a in b or b in a


def cruzar_card_amrfinder(rgi, amrfinder):
    """Cruza los ARG detectados por CARD/RGI y AMRFinderPlus, por gen especifico y por
    familia/clase de droga. Heuristico, primera aproximacion -- no hay mapeo oficial 1:1 entre
    las dos nomenclaturas."""
    genes_card = sorted(rgi["Best_Hit_ARO"].dropna().unique())
    genes_amr = sorted(amrfinder["Element symbol"].dropna().unique())

    filas, vistos_amr = [], set()
    for g_card in genes_card:
        candidatos = [g for g in genes_amr if mismo_gen(g_card, g)]
        vistos_amr.update(candidatos)
        filas.append({"gen": g_card, "en_CARD": True, "en_AMRFinderPlus": bool(candidatos),
                      "gen_AMRFinderPlus": ", ".join(candidatos) or None})
    for g_amr in genes_amr:
        if g_amr not in vistos_amr:
            filas.append({"gen": g_amr, "en_CARD": False, "en_AMRFinderPlus": True, "gen_AMRFinderPlus": g_amr})
    cruce_gen = pd.DataFrame(filas)

    familias_card = sorted(rgi["Drug Class"].dropna().str.split(";").explode().str.strip().unique())
    clases_amr = sorted(amrfinder["Class"].dropna().unique())

    filas_fam, vistas_amr = [], set()
    for f_card in familias_card:
        candidatas = [c for c in clases_amr if mismo_gen(f_card, c)]
        vistas_amr.update(candidatas)
        filas_fam.append({"familia": f_card, "en_CARD": True, "en_AMRFinderPlus": bool(candidatas)})
    for c_amr in clases_amr:
        if c_amr not in vistas_amr:
            filas_fam.append({"familia": c_amr, "en_CARD": False, "en_AMRFinderPlus": True})
    cruce_familia = pd.DataFrame(filas_fam)

    ambos = (cruce_gen["en_CARD"] & cruce_gen["en_AMRFinderPlus"]).sum()
    solo_card = (cruce_gen["en_CARD"] & ~cruce_gen["en_AMRFinderPlus"]).sum()
    solo_amr = (~cruce_gen["en_CARD"] & cruce_gen["en_AMRFinderPlus"]).sum()
    print(f"Cruce CARD/AMRFinderPlus -- genes: {ambos} en ambas, {solo_card} solo CARD, "
          f"{solo_amr} solo AMRFinderPlus ({len(cruce_familia)} familias/clases distintas)")
    return cruce_gen, cruce_familia


def graficar_resistoma(rgi, run):
    fam = rgi["AMR Gene Family"].value_counts()
    ax = fam.plot(kind="barh", figsize=(7, max(2, 0.3 * len(fam))))
    ax.invert_yaxis()
    plt.xlabel("Nº de genes detectados")
    plt.title(f"Familias de genes de resistencia (CARD) — {run}")
    plt.tight_layout()
    plt.show()

### 3.5 Posición de cada ARG dentro de su contig (CARD/RGI y AMRFinderPlus)

Las dos herramientas ya traen coordenadas de nucleótido dentro de su contig por gen -- RGI las toma
del encabezado que deja Prodigal en `genes.faa`; AMRFinderPlus las reporta directamente (`Contig
id`/`Start`/`Stop`/`Strand`) porque corre en modo combinado (sección 3.4). En un metagenoma
ensamblado no existe una única coordenada "de la muestra": cada contig es un fragmento
independiente, así que la ubicación más fina posible es *(contig, posición dentro de ese contig)*.
Añadimos el largo del contig y la posición relativa (0 = inicio del contig, 1 = final) para poder
comparar genes entre contigs de distinto tamaño, y para detectar genes truncados por quedar muy
cerca del borde de un contig corto (típico de ensamblados fragmentados).

In [ ]:
def contig_de_orf(orf, asm_ids):
    """Deduce el contig real del ORF_ID de RGI, anclandose en los IDs del ensamblado."""
    orf = str(orf).split()[0]
    if orf in asm_ids:
        return orf
    partes = orf.split("_")
    for corte in range(len(partes) - 1, 0, -1):
        cand = "_".join(partes[:corte])
        if cand in asm_ids:
            return cand
    return None


def agregar_posicion_relativa(df, contigs_path, col_contig, col_start, col_stop, resolver_contig=None):
    """Añade largo de contig y posición relativa (0-1) a cada fila de un dataframe de ARGs.
    resolver_contig hace falta para RGI (su ORF_ID no siempre coincide 1:1 con el ID del contig);
    AMRFinderPlus ya reporta el contig real en 'Contig id', asi que ahi se usa tal cual (resolver_contig=None)."""
    largos = {r.id: len(r.seq) for r in SeqIO.parse(str(contigs_path), "fasta")}
    df = df.copy()
    if resolver_contig:
        asm_ids = set(largos)
        df["contig_asm"] = df[col_contig].map(lambda x: resolver_contig(x, asm_ids))
    else:
        df["contig_asm"] = df[col_contig]
    df["contig_len"] = df["contig_asm"].map(largos)
    df["pos_inicio_rel"] = df[col_start] / df["contig_len"]
    df["pos_fin_rel"] = df[col_stop] / df["contig_len"]

    sin_contig = df["contig_asm"].isna().sum()
    if sin_contig:
        print(f"Aviso: {sin_contig}/{len(df)} genes sin contig identificado en el ensamblado")
    return df

### 3.6 Limpieza: borrar crudos e intermedios

A diferencia de versiones anteriores, la vinculación a especie/posición de referencia (sección 3.7)
ya no queda diferida a mano -- corre en línea, dentro de `procesar_muestra()`, antes de limpiar. Por
eso ya no hace falta conservar nada de `work/<run>/` (ni siquiera los contigs) después de terminar
una muestra: todo lo que se necesita después (`05_modelo_ml.ipynb`, o reanudar una corrida cortada)
queda persistido en `results/<run>/` -- `resistoma_<run>.csv` (la tabla enriquecida: CARD +
AMRFinderPlus + posición + especie + posición de referencia) y los outputs nativos de cada
herramienta (`rgi_<run>.txt`, `amrfinder_<run>.tsv`).

In [ ]:
def limpiar_muestra(run, raw_dir, work_dir):
    """Borra las lecturas crudas y todos los intermedios de una muestra ya procesada.

    Ya no hace falta conservar nada de work/<run>/ (ni siquiera los contigs) -- la vinculacion a
    especie/posicion de referencia ya corrio en linea dentro de procesar_muestra() antes de llegar
    aqui (seccion 3.7), asi que no queda diferida a una corrida posterior que necesite releerlos.
    Todo lo que hace falta despues (05_modelo_ml.ipynb, o reanudar una corrida cortada) esta en
    results/<run>/, que esta funcion no toca.
    """
    liberado = 0
    for ruta in (raw_dir, work_dir):
        if not ruta.exists():
            continue
        liberado += sum(f.stat().st_size for f in ruta.rglob("*") if f.is_file())
        shutil.rmtree(ruta, ignore_errors=True)

    print(f"Limpieza: liberados {_human(liberado)} de {run} (se conserva results/{run}/ completo)")

### 3.7 Vincular cada ARG a su especie y posición genómica (BLAST local)

Ya **no** es manual/diferido: `procesar_muestra()` la llama automáticamente para cada muestra,
justo antes de la limpieza (sección 3.6). Lo que antes hacía este paso poco práctico de correr por
muestra era el límite de tasa de BLAST remoto contra NCBI -- con la base `nt` local eso desaparece
(sección 2), así que se vuelve un paso más del pipeline, no una excepción a mano.

Vincula por especie **y** proyecta la posición del ARG sobre la secuencia de referencia que BLAST
devolvió como mejor hit (`ref_pos_inicio`/`ref_pos_fin`, junto con `hit_accession`): interpolación
lineal sobre el tramo que BLAST alineó, válida solo si el ARG cae dentro de ese tramo, y sujeta a
que el accession devuelto sea un genoma completo o solo un registro génico corto (depende de lo que
haya depositado esa especie en `nt`). Corre **una sola vez por muestra**, sobre la unión de contigs
con ARG de CARD y AMRFinderPlus (no dos búsquedas separadas), y enriquece los dos dataframes con el
mismo resultado.

BacDive sigue fuera de aquí (se mueve a `05_modelo_ml.ipynb`, para consultarla solo por las
especies de los genes que el modelo termine señalando).

In [ ]:
def vincular_arg_a_especie(rgi, amrfinder, contigs, out_dir, tag):
    """Vincula los ARG de CARD y AMRFinderPlus a especie y posicion de referencia, con una sola
    tanda de blastn local sobre la union de contigs de ambas herramientas (asume que ya tienen
    'contig_asm', 'Start' y 'Stop' -- ver agregar_posicion_relativa, seccion 3.5)."""
    contigs_por_id = {r.id: r for r in SeqIO.parse(str(contigs), "fasta")}

    wanted = set(rgi["contig_asm"].dropna()) | set(amrfinder["contig_asm"].dropna())
    recs = [contigs_por_id[c] for c in wanted if c in contigs_por_id]
    print(f"Contigs con ARG (CARD + AMRFinderPlus) a identificar por BLAST local: {len(recs)}")

    blast_df = blast_local_taxonomia_y_posicion(recs, out_dir, tag)

    def enriquecer(df):
        df = df.merge(blast_df, left_on="contig_asm", right_on="contig", how="left").drop(columns="contig")
        if len(df):
            df[["ref_pos_inicio", "ref_pos_fin"]] = df.apply(proyectar_posicion_en_referencia, axis=1)
        else:
            df["ref_pos_inicio"], df["ref_pos_fin"] = None, None
        return df

    return enriquecer(rgi), enriquecer(amrfinder), blast_df


def enriquecer_con_bacdive(rgi_con_taxon):
    especies = sorted(rgi_con_taxon["taxon"].dropna().unique())
    print(f"Especies distintas a consultar en BacDive: {len(especies)}")
    rasgos = pd.DataFrame([bacdive_traits(sp) for sp in especies])
    return rgi_con_taxon.merge(rasgos, on="taxon", how="left")

### 3.8 Orquestación: pipeline completo para una muestra

In [ ]:
def procesar_muestra(run):
    raw_dir, work_dir, out_dir = Path(f"raw/{run}"), Path(f"work/{run}"), Path(f"results/{run}")
    resistoma_out = out_dir / f"resistoma_{run}.csv"
    if resistoma_out.exists():
        print(f"\n{run}: ya procesada y limpiada en una corrida anterior, se omite")
        return pd.read_csv(resistoma_out)

    print(f"\n{'=' * 60}\nProcesando muestra: {run}\n{'=' * 60}")
    for d in (raw_dir, work_dir, out_dir):
        d.mkdir(parents=True, exist_ok=True)

    r1, r2 = resolver_y_descargar(run, raw_dir)
    sub1, sub2 = qc_y_submuestreo(run, r1, r2, work_dir, out_dir)
    contigs, genes_faa, genes_gff = ensamblar_y_predecir_genes(sub1, sub2, work_dir)

    rgi = detectar_arg(run, genes_faa, out_dir)
    amrfinder = detectar_arg_amrfinder(run, contigs, genes_faa, genes_gff, out_dir)
    cruce_gen, cruce_familia = cruzar_card_amrfinder(rgi, amrfinder)
    cruce_gen.to_csv(out_dir / f"cruce_gen_{run}.csv", index=False)
    cruce_familia.to_csv(out_dir / f"cruce_familia_{run}.csv", index=False)

    rgi = agregar_posicion_relativa(rgi, contigs, "ORF_ID", "Start", "Stop", resolver_contig=contig_de_orf)
    amrfinder = agregar_posicion_relativa(amrfinder, contigs, "Contig id", "Start", "Stop")

    rgi, amrfinder, _ = vincular_arg_a_especie(rgi, amrfinder, contigs, out_dir, run)

    graficar_resistoma(rgi, run)

    rgi.insert(0, "herramienta", "CARD/RGI")
    amrfinder.insert(0, "herramienta", "AMRFinderPlus")
    resistoma = pd.concat([rgi, amrfinder], ignore_index=True)
    resistoma.insert(0, "run", run)
    resistoma.to_csv(resistoma_out, index=False)

    limpiar_muestra(run, raw_dir, work_dir)
    return resistoma

## 4. Ejecutar sobre las muestras configuradas

In [ ]:
resultados_por_muestra = {}
fallas = {}
for run in RUNS:
    try:
        resultados_por_muestra[run] = procesar_muestra(run)
    except Exception as e:
        print(f"\n!! {run} fallo, se continua con la siguiente muestra: {type(e).__name__}: {e}\n")
        fallas[run] = e

if fallas:
    print(f"\nMuestras que fallaron ({len(fallas)}/{len(RUNS)}): {list(fallas.keys())}")

resultados_totales = pd.concat(resultados_por_muestra.values(), ignore_index=True)
print(f"\nTotal de genes de resistencia detectados, todas las muestras (CARD + AMRFinderPlus): "
      f"{len(resultados_totales)}")
resultados_totales[["run", "herramienta", "Best_Hit_ARO", "Element symbol", "Drug Class", "Class",
                     "contig_asm", "Start", "Stop", "contig_len", "pos_inicio_rel", "pos_fin_rel",
                     "taxon", "identidad_%", "hit_accession", "ref_pos_inicio", "ref_pos_fin"]]

## 5. Propuesta de modelo

Este pipeline muestra el proceso de obtención de datos para etiquetado, entrenamiento y testeo de un modelo de Machine Learning con usos prácticos para el problema de detección de resistencia a antibióticos. La propuesta de diseño y construcción de este modelo, es un modelo que, leyendo secuencias de DNA, como las que se encuentran de manera cruda en la base de datos de ENA, pueda predecir si contiene genes de resistencia bacteriana, y si los tiene, cuáles son. El etiquetado y verificación sería consultar los genes que retorne el modelo, y verificar si en efecto son genes de resistencia. El aspecto novedoso del experimento, es ver si el modelo puede identificar correctamente genes de resistencia, aunque no hagan parte del conjunto de entrenamiento. Es decir, ver si el modelo puede correctamente detectar más allá de una lista de genes de resistencia conocidos. Si el modelo tiene una capacidad predictiva, y no solo de detección de patrones conocidos, puede servir como un punto de partida para detección de patrones con predisposición a desarrollar resistencia en el futuro. 

## 6. Resumen de ARGs detectados hasta ahora

Lee directamente de `results/<run>/resistoma_<run>.csv` en disco (no depende de que la celda de la sección 4
haya terminado de correr) para resumir lo detectado en todas las muestras ya completamente procesadas. Ya
viene enriquecido (posición, especie, posición de referencia) -- no hace falta reconstruir nada a partir de
los contigs, que ya no se conservan (sección 3.6).

In [ ]:
def cargar_resultados_disponibles():
    """Carga todos los resistoma_<run>.csv ya completados en results/, leyendo de disco en vez de
    depender del estado en memoria de la celda de ejecucion (que puede seguir corriendo)."""
    filas = []
    for csv_path in sorted(Path("results").glob("*/resistoma_*.csv")):
        if csv_path.stat().st_size == 0:
            continue
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        filas.append(df)
    return pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()


resultados_disponibles = cargar_resultados_disponibles()
n_muestras = resultados_disponibles["run"].nunique() if not resultados_disponibles.empty else 0
print(f"Muestras con resultados completos: {n_muestras}")
print(f"Total de genes de resistencia detectados (CARD + AMRFinderPlus): {len(resultados_disponibles)}")

# El resumen de familias/clases se hace sobre CARD -- AMRFinderPlus usa su propia nomenclatura
# (Element symbol/Class, sin mapeo 1:1 con Best_Hit_ARO/Drug Class; ver cruce_gen_<run>.csv por
# muestra para la correspondencia heuristica entre las dos).
card_disponible = resultados_disponibles[resultados_disponibles["herramienta"] == "CARD/RGI"]
tabla_arg = (
    card_disponible
    .groupby(["Best_Hit_ARO", "AMR Gene Family", "Drug Class"])
    .agg(n_detecciones=("run", "size"), n_muestras=("run", "nunique"))
    .sort_values(["n_muestras", "n_detecciones"], ascending=False)
    .reset_index()
)
display(tabla_arg)

top = tabla_arg.head(15).sort_values("n_muestras")
fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(top))))
ax.barh(top["Best_Hit_ARO"], top["n_muestras"], color="#3B7EA1")
ax.set_xlabel("Nº de muestras en las que se detectó")
ax.set_title(f"ARGs (CARD) más extendidos entre las {n_muestras} muestras procesadas")
plt.tight_layout()
plt.show()